In [1]:
import os
import pickle
import numpy as np
import pandas as pd

import scanpy as sc
import mudata

from scenicplus.data_wrangling.adata_cistopic_wrangling import process_multiome_data

/opt/conda/envs/scenicplus/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-08-14 05:03:27,655	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
data_dir = "/scenic_plus_vol/SCENIC/full_dataset"
cisTopic_obj_fname = f"{data_dir}/cistopic_obj.pkl"
GEX_anndata_fname = f"{data_dir}/GEX.h5ad"

In [ ]:
GEX_anndata = mudata.read(GEX_anndata_fname.__str__())

with open(cisTopic_obj_fname, "rb") as f:
    cisTopic_obj = pickle.load(f)

In [ ]:
all_cells = GEX_anndata.obs.index.values
shuffled_cells = np.random.permutation(all_cells)

# Split cells into 4 (approximately) equally sized groups
cells_per_replicate = np.array_split(shuffled_cells, 4)
for rep in range(len(cells_per_replicate)):
    out_dir = f"{data_dir}/downsampled_rep{rep+1}"
    os.mkdir(out_dir)
    pd.Series(cells_per_replicate[rep]).to_csv(f"{out_dir}/cells.tsv", index=False, header=False)

In [ ]:
bc_transform_func = lambda x: f'{x}___cisTopic'

for rep in range(len(cells_per_replicate)):
    print(rep)
    out_dir = f"{data_dir}/downsampled_rep{rep+1}"
    out_file = f"{out_dir}/ACC_GEX.h5mu"

    # manually process rna+atac into combined object
    # although this step is also part of the snakemake pipeline,
    # this saves some storage space by not separately saving the rna and atac subsets
    cells = pd.read_csv(f"{out_dir}/cells.tsv", header=None)[0].values
    GEX_anndata_subset = GEX_anndata[cells,:]
    mdata = process_multiome_data(
        GEX_anndata=GEX_anndata_subset,
        cisTopic_obj=cisTopic_obj,
        use_raw_for_GEX_anndata=True,
        bc_transform_func=bc_transform_func)
    mdata.write_h5mu(out_file)

    del mdata
    del GEX_anndata_subset

2
2025-08-14 02:53:21,219 Ingesting multiome data INFO     Found 215851 multiome cells.
2025-08-14 02:53:21,992 cisTopic     INFO     Imputing region accessibility
2025-08-14 02:53:21,993 cisTopic     INFO     Impute region accessibility for regions 0-20000
2025-08-14 02:53:30,041 cisTopic     INFO     Impute region accessibility for regions 20000-40000
2025-08-14 02:53:38,090 cisTopic     INFO     Impute region accessibility for regions 40000-60000
2025-08-14 02:53:46,129 cisTopic     INFO     Impute region accessibility for regions 60000-80000
2025-08-14 02:53:54,132 cisTopic     INFO     Impute region accessibility for regions 80000-100000
2025-08-14 02:54:02,157 cisTopic     INFO     Impute region accessibility for regions 100000-120000
2025-08-14 02:54:10,191 cisTopic     INFO     Impute region accessibility for regions 120000-140000
2025-08-14 02:54:18,215 cisTopic     INFO     Impute region accessibility for regions 140000-160000
2025-08-14 02:54:26,257 cisTopic     INFO     Imp

... storing 'sample_id' as categorical
... storing 'Chromosome' as categorical


3
2025-08-14 03:40:07,764 Ingesting multiome data INFO     Found 215850 multiome cells.
2025-08-14 03:40:08,385 cisTopic     INFO     Imputing region accessibility
2025-08-14 03:40:08,386 cisTopic     INFO     Impute region accessibility for regions 0-20000
2025-08-14 03:40:16,397 cisTopic     INFO     Impute region accessibility for regions 20000-40000
2025-08-14 03:40:24,388 cisTopic     INFO     Impute region accessibility for regions 40000-60000
2025-08-14 03:40:32,374 cisTopic     INFO     Impute region accessibility for regions 60000-80000
2025-08-14 03:40:40,329 cisTopic     INFO     Impute region accessibility for regions 80000-100000
2025-08-14 03:40:48,308 cisTopic     INFO     Impute region accessibility for regions 100000-120000
2025-08-14 03:40:56,301 cisTopic     INFO     Impute region accessibility for regions 120000-140000
2025-08-14 03:41:04,290 cisTopic     INFO     Impute region accessibility for regions 140000-160000
2025-08-14 03:41:12,299 cisTopic     INFO     Imp

... storing 'sample_id' as categorical
... storing 'Chromosome' as categorical
